# 03 • Tipos, ausências e qualidade

## Preparar esta sessão
Execute as células abaixo antes de começar a aula. Elas localizam os arquivos e preparam os objetos necessários. Você não precisa decorar essa preparação agora.

In [ ]:
from pathlib import Path
import pandas as pd
import math
BASE = Path.cwd()
if not (BASE / 'dados').exists() and (BASE.parent / 'dados').exists():
    BASE = BASE.parent
DADOS = BASE / 'dados'
SAIDAS = BASE / 'resultados'
SAIDAS.mkdir(exist_ok=True)
assert (DADOS/'penguins.csv').exists(), 'Descompacte o curso inteiro e abra na pasta correta.'


## Preparação independente
Execute esta célula para recriar os objetos necessários dos módulos anteriores.

In [ ]:
p = pd.read_csv(DADOS / 'penguins.csv')
print(p.head(3).to_string(index=False))
print('Dimensões:', p.shape)
print('Colunas:', p.columns.tolist())
print(type(p['species']).__name__)

mascara = p['species'].eq('Adelie') & p['body_mass_g'].gt(4000)
print(p.loc[mascara, ['species', 'island', 'body_mass_g']].head())
print(p.iloc[:3, :2])

### Objetivo: distinguir valor conhecido, valor ausente e valor inválido

Uma tabela pode abrir sem erros e ainda conter dados inadequados para uma análise. Nesta aula, vamos responder: “em quais colunas faltam dados?” e “quais linhas posso usar para estudar a massa corporal?”.

### 1. O que significa um valor ausente?


In [ ]:
print(p[['species', 'body_mass_g']].head(4))


Na quarta observação, a massa aparece como NaN. Isso indica ausência de uma medição nessa coluna. Não significa que o animal pesa zero e não autoriza inventar sua massa. Em outras colunas ou tipos, a ausência pode aparecer como None, <NA> ou NaT. Use isna para detectar ausências; não tente comparar com o texto 'NaN'.


In [ ]:
falta_massa = p['body_mass_g'].isna()
print(falta_massa.head(4))
print(falta_massa.sum())


O começo da condição é False, False, False, True. A soma conta True como 1 e False como 0. O total da base é 2: há duas observações sem massa. Isso deixa 342 medições presentes, não 344.

### 2. De contagem a porcentagem


In [ ]:
quantidade = falta_massa.sum()
total = len(p)
percentual = quantidade / total * 100
print(quantidade, total, round(percentual, 2))


len(p) é a quantidade de linhas. A conta é 2 dividido por 344, multiplicado por 100: aproximadamente 0,58%. round(..., 2) arredonda apenas o número apresentado para duas casas. Outra forma é `falta_massa.mean() * 100`, porque a média dos zeros e uns representa a proporção de ausências.


In [ ]:
print(p.isna().sum())


Aplicado à tabela inteira, isna cria uma tabela de respostas; sum conta por coluna. Nesta base, sex tem 11 ausências e body_mass_g tem 2. Depois de entender isso, podemos reunir contagem e porcentagem em um relatório, como no exemplo guiado ao final.

### 3. Remover somente o que a pergunta exige


In [ ]:
com_massa = p.dropna(subset=['body_mass_g']).copy()
print(len(com_massa))


dropna remove linhas com ausência nas colunas indicadas em subset. O resultado tem 342 linhas. A chamada copy deixa explícito que trabalharemos com uma tabela separada. A base p permanece com 344 linhas. Não use simplesmente `p.dropna()` sem pensar: isso exige informação presente em todas as colunas e pode descartar animais por falta de sexo, mesmo quando você só quer estudar massa.

Se a pergunta envolver massa e nadadeira, use `subset=['body_mass_g', 'flipper_length_mm']`. Documente a perda de observações. Preencher por média ou mediana é uma escolha metodológica, não uma etapa obrigatória de limpeza; não faremos isso automaticamente.

### 4. Tipos: o que a coluna permite fazer?


In [ ]:
print(p.dtypes)


dtypes informa o tipo de cada coluna. O tipo numérico permite fazer contas. Um tipo de texto guarda nomes ou códigos; sua aparência exata pode variar entre versões. Um número que veio como texto precisa ser convertido antes de uma operação numérica. Porém, um código como '0012' deve continuar texto para preservar seus zeros e seu significado. Ter apenas dígitos não transforma um identificador em medida.

Para listar colunas reconhecidas como numéricas, use `p.select_dtypes(include='number').columns`. Essa operação seleciona por tipo, não por significado: ainda cabe a você decidir se faz sentido calcular uma média daquela coluna.

### 5. Converter e contar os problemas


In [ ]:
entrada = pd.Series(['10', 'erro', None])
numeros = pd.to_numeric(entrada, errors='coerce')
print(numeros)


Este é um exemplo fictício de importação problemática. O primeiro valor vira 10. O texto erro não pode virar número. Com errors='coerce', ele vira ausência em vez de interromper a conversão. A ausência original também continua ausente. Ao final há duas ausências, mas só uma foi criada por falha de conversão:


In [ ]:
novas_falhas = entrada.notna() & numeros.isna()
print(novas_falhas.sum())


notna é o contrário de isna. A condição procura valores que estavam presentes antes e ficaram ausentes depois. O resultado esperado é 1. Usar coerce sem medir esse efeito pode esconder erros importantes. astype é útil quando você já sabe que a conversão deve funcionar, como `.astype('Int64')` para inteiros que aceitam ausências. Datas terão uma aula própria.

### 6. Duplicatas exigem uma definição


In [ ]:
print(p.duplicated().sum())


duplicated marca linhas iguais a uma anterior, considerando todas as colunas por padrão. O total encontrado não é uma autorização para apagar. Dois animais podem ter as mesmas medidas registradas, e a base simplificada não tem um identificador individual explícito. drop_duplicates remove essas repetições, mas só deve ser usado depois de justificar o que caracteriza um registro duplicado no seu problema.

### O que guardar desta aula

Ausência não é zero; converter não é corrigir; remover exige justificativa. Antes de uma limpeza, registre o tamanho original. Depois, registre quantas linhas foram removidas e por qual motivo. Preserve o arquivo bruto para conseguir revisar suas decisões.


## Exemplo guiado

In [ ]:
qualidade = pd.DataFrame({'ausentes':p.isna().sum(), 'percentual':p.isna().mean().mul(100)})
print(qualidade)
medidos = p.dropna(subset=['body_mass_g']).copy()
print('Observações com massa:', len(medidos))

## Exercício 03.1
Quais colunas possuem pelo menos uma ausência?

In [ ]:
# Escreva sua resposta aqui


## Exercício 03.2
Calcule a porcentagem de massas ausentes.

In [ ]:
# Escreva sua resposta aqui


## Exercício 03.3
Crie uma tabela apenas com massa e comprimento de nadadeira presentes. Conte a perda de linhas.

In [ ]:
# Escreva sua resposta aqui


## Exercício 03.4
Converta a Series didática ["10", "erro", None] para número e conte ausências resultantes.

In [ ]:
# Escreva sua resposta aqui


## Exercício 03.5
Conte duplicatas exatas, sem removê-las. Por que isso não basta para excluir observações?

In [ ]:
# Escreva sua resposta aqui
